In [0]:
%pip install openpyxl==3.1.2 "mlflow[databricks]" "pydantic>=1.10,<2"

In [0]:
import pyspark.sql.functions as fn
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import KBinsDiscretizer, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import mlflow
import numpy as np
import pandas as pd

In [0]:
table_name = "hbse.default.sixers_rfm_dataset"

# read aggregated data
primary_rev = (
  spark.table(table_name)
      .withColumn('total_revenue', fn.round(fn.col('total_sales') * fn.col('total_seats'), 2)) # product of sales and seats
  )

# display dataset
display(primary_rev)

In [0]:
primary_rev_consolidated = (
primary_rev
    .filter('total_revenue > 0.0') # remove returns from dataset
    .withColumn('saledate', fn.to_date('saledate')) # convert date to just datepart
    .groupBy('audienceid', 'saledate', 'ledgername') # group on customer and date
      .agg(fn.sum('total_revenue').alias('total_revenue')) # sum sales amount
  )

display(primary_rev_consolidated)

In [0]:
# get last date in dataset
last_date = (
  primary_rev_consolidated
    .groupBy()
      .agg(fn.max('saledate').alias('lastdate'))
  )

# calculate metrics
rfm_metrics = (
  primary_rev_consolidated
    .crossJoin(last_date)
    .groupBy('audienceid') # for each customer
      .agg(
        fn.min(fn.datediff('lastdate','saledate')).alias('recency'), # days since last date in dataset (get lowest value as recency)
        fn.countDistinct('saledate').alias('frequency'), # unique dates on which purchases occur
        fn.round(fn.avg('total_revenue'), 2).alias('monetary_value') # avg spend per purchase, rounded to 2 decimal places (works because we've already summed sales per customer-date)
        )
  )

# display metrics
display(
  rfm_metrics
)

In [0]:
# send metrics to pandas for visualizations
df_pd = rfm_metrics.toPandas()

# quantiles for each metric, to complement the histograms below with numeric detail
display(df_pd[['recency', 'frequency', 'monetary_value']].quantile([.1, .25, .5, .75, .9, 1.0]))

# configure plot as three charts in a single row
f, axes = plt.subplots(nrows=1, ncols=3, squeeze=True, figsize=(32,10))

# generate one chart per metric
for i, metric in enumerate(['recency', 'frequency', 'monetary_value']):
   
  # use metric name as chart title
  axes[i].set_title(metric)
  
  # define histogram chart
  axes[i].hist(df_pd[metric], bins=10)

In [0]:
# help decide a data-driven cap for recency (days since last purchase)
print('recency high percentiles:')
print(df_pd['recency'].describe(percentiles=[.75, .9, .95, .975, .99, .995, .999]))

print('\ncustomers affected by candidate caps:')
for cap in [365, 730, 1095, 1460, 1825, 2500, 3650, 5000]:
  n_above = (df_pd['recency'] > cap).sum()
  pct_above = n_above / len(df_pd) * 100
  print(f'  cap={cap:>5} days (~{cap/365:.1f} yrs): {n_above:>5} customers above ({pct_above:.2f}%)')

# IQR-based outlier boundary, for reference
q1, q3 = df_pd['recency'].quantile([.25, .75])
iqr = q3 - q1
print(f"\nIQR-based outlier boundary (Q3 + 1.5*IQR): {q3 + 1.5*iqr:,.0f} days")
print(f"IQR-based outlier boundary (Q3 + 3*IQR):   {q3 + 3*iqr:,.0f} days")

In [0]:
# help decide a data-driven cap for monetary_value instead of the current hardcoded $3000
print('monetary_value high percentiles:')
print(df_pd['monetary_value'].describe(percentiles=[.75, .9, .95, .975, .99, .995, .999]))

pct_at_or_below_3000 = (df_pd['monetary_value'] <= 3000).mean() * 100
print(f"\nthe current $3000 cap sits at the {pct_at_or_below_3000:.1f}th percentile "
      f"({100 - pct_at_or_below_3000:.1f}% of customers get capped)")

print('\ncustomers affected by candidate caps:')
for cap in [1500, 3000, 5000, 10000, 20000, 50000, 100000]:
  n_above = (df_pd['monetary_value'] > cap).sum()
  pct_above = n_above / len(df_pd) * 100
  print(f'  cap=${cap:>7,}: {n_above:>5} customers above ({pct_above:.2f}%)')

# IQR-based outlier boundary, for reference
q1, q3 = df_pd['monetary_value'].quantile([.25, .75])
iqr = q3 - q1
print(f"\nIQR-based outlier boundary (Q3 + 1.5*IQR): ${q3 + 1.5*iqr:,.2f}")
print(f"IQR-based outlier boundary (Q3 + 3*IQR):   ${q3 + 3*iqr:,.2f}")

In [0]:
# calculate metrics
rfm_metrics_cleansed = (
  rfm_metrics
    .withColumn('recency', fn.expr("case when recency > 1260 then 1260 else recency end"))
    .withColumn('frequency', fn.expr("case when frequency > 5 then 5 else frequency end"))
    .withColumn('monetary_value', fn.expr("case when monetary_value > 5000 then 5000 else monetary_value end"))
  )

In [0]:
# extract metrics to pandas for visualization
df_pd = rfm_metrics_cleansed.toPandas()

# configure plot as three charts in a single row
f, axes = plt.subplots(nrows=1, ncols=3, squeeze=True, figsize=(32,10))

# generate one chart per metric
for i, metric in enumerate(['recency', 'frequency', 'monetary_value']):
   
  # use metric name as chart title
  axes[i].set_title(metric)
  
  # define chart
  axes[i].hist(df_pd[metric], bins=10)

In [0]:
inputs_pd = rfm_metrics_cleansed.toPandas()

In [0]:
# defining binning transformation
binner = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')

# apply binner to recency and monetary only
# frequency is binary encoded in the next cell
col_trans = ColumnTransformer(
  [
    ('r_bin', binner, ['recency']),
    ('m_bin', binner, ['monetary_value'])
    ],
  remainder='drop'
  )

In [0]:
# invert the recency values so that higher is better
inputs_pd['recency'] = inputs_pd['recency'] * -1

# bin the data
bins = col_trans.fit_transform(inputs_pd)

# add bins to input data
inputs_pd['r_bin'] = bins[:,0]
inputs_pd['m_bin'] = bins[:,1]

# binary encode frequency: 0 = bought once, 1 = bought more than once
inputs_pd['f_bin'] = (inputs_pd['frequency'] > 1).astype(float)

# display dataset
display(inputs_pd)

In [0]:
# consolidated diagnostics: examine skew in recency and frequency, and the resulting bin/gap behavior

# recency (after -1 inversion): unique values + top value counts
print(f"Unique recency values: {inputs_pd['recency'].nunique()}")
print(f"Total customers: {len(inputs_pd)}")
print(f"\nRecency value counts (top 20):")
print(inputs_pd['recency'].value_counts().head(20).to_string())

# frequency: unique values, value counts, and the resulting f_bin distribution
print(f"\nUnique frequency values: {inputs_pd['frequency'].nunique()}")
print(f"\nFrequency value counts (top 20):")
print(inputs_pd['frequency'].value_counts().head(20).to_string())
print(f"\nf_bin distribution:")
print(inputs_pd['f_bin'].value_counts().sort_index().to_string())

# recency histogram + describe, to diagnose the gap in values noted above
counts, edges = np.histogram(inputs_pd['recency'], bins=10)
print("\nRecency histogram (raw values currently in inputs_pd):")
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"  [{lo:8.1f}, {hi:8.1f}) -> {c}")

print("\nrecency describe:")
print(inputs_pd['recency'].describe())

# check underlying sale activity by month, to explain the gap via a lull period
sale_activity = (
  primary_rev_consolidated
    .groupBy(fn.date_trunc('MONTH', 'saledate').alias('month'))
    .agg(fn.count('*').alias('n_transactions'))
    .orderBy('month')
  )
display(sale_activity)

In [0]:
# configure plot as three charts in a single row
f, axes = plt.subplots(nrows=1, ncols=3, squeeze=True, figsize=(32,10))

for i, metric in enumerate(['r_bin','f_bin','m_bin']):
   
  # use metric name as chart title
  axes[i].set_title(metric)
  
  # define chart
  axes[i].hist(inputs_pd[metric], bins=5)

In [0]:
# show the underlying metric range (min/max) captured by each r_bin, f_bin, and m_bin value
for bin_col, metric in [('r_bin', 'recency'), ('f_bin', 'frequency'), ('m_bin', 'monetary_value')]:

  df = inputs_pd[[bin_col, metric]].copy()

  # recency is stored inverted (*-1) for binning purposes; restore positive days-since-purchase for readability
  if metric == 'recency':
    df[metric] = df[metric].abs()

  ranges = (
    df
      .groupby(bin_col)[metric]
      .agg(min_value='min', max_value='max', customers='count')
      .sort_index()
      .reset_index()
    )

  print(f"\n{bin_col} ranges (based on {metric}):")
  display(ranges)

In [0]:
# train the tsne model and compute x and y axes for our values 
# (adjust the perplexity value higher or lower until you receive a visual result that's meaningful)
tsne = TSNE(n_components=2, perplexity=80, init='pca', learning_rate='auto')
tsne_results = tsne.fit_transform(inputs_pd[['r_bin','f_bin','m_bin']])

# return the axes assignments to our metrics dataset
inputs_pd['tsne_one'] = tsne_results[:,0]
inputs_pd['tsne_two'] = tsne_results[:,1]

# display results
display(inputs_pd)

In [0]:
# configure plot as three charts in a single row
f, axes = plt.subplots(nrows=1, ncols=3, squeeze=True, figsize=(32,10))

for i, metric in enumerate(['r_bin', 'f_bin', 'm_bin']):
  
  # unique values for this metric
  n = inputs_pd[['{0}'.format(metric)]].nunique()[0]
  
  # use metric name as chart title
  axes[i].set_title(metric)
  
  # define chart
  sns.scatterplot(
    x='tsne_one',
    y='tsne_two',
    hue='{0}'.format(metric),
    palette=sns.color_palette('coolwarm', n),
    data=inputs_pd,
    legend=False,
    alpha=0.4,
    ax = axes[i]
    )

In [0]:
# define max value of k to explore
max_k = 15

# reference the pandas DataFrame for the UDF closure (serialized automatically)
_inputs_data = inputs_pd[['r_bin','f_bin','m_bin']].copy()


# function to train and score clusters based on k cluster count
@fn.udf('float')
def get_silhouette(k):

  # train a model on k
  km = KMeans(
    n_clusters=k, 
    init='random',
    n_init=10000
    )
  kmeans = km.fit( _inputs_data )

  # get silhouette score for model (sample_size bounds memory)
  silhouette = silhouette_score( 
      _inputs_data,  # x values
      kmeans.predict(_inputs_data), # cluster assignments 
      sample_size=2000
      )
  
  # return score
  return float(silhouette)


# assemble an dataframe containing each k value
iterations = (
  spark
    .range(2, max_k + 1, step=1) # get values for k
    .withColumnRenamed('id','k') # rename to k
    .repartition( max_k-1, 'k' ) # ensure data are well distributed
    .withColumn('silhouette', get_silhouette('k'))
  )
  
# display the results of our analysis
display( 
  iterations
      )

In [0]:
# define model
model = KMeans(
  n_clusters=4, 
  init='random',
  n_init=10000
  )

# couple model with transformations
pipe = Pipeline(steps=[
  ('binnerize', col_trans),
  ('cluster', model)
  ])

# train pipeline
fitted_pipe = pipe.fit( inputs_pd )

# assign clusters
inputs_pd['cluster'] = pipe.predict( inputs_pd )

# display cluster assignments
display(inputs_pd)

In [0]:
# composite RFM score: weighted combination of recency, frequency, and monetary value
# note: 'recency' in inputs_pd is stored inverted (multiplied by -1 in "Apply Binning Logic" above),
# so higher (less negative) values already indicate more recent purchases. Multiplying by a
# positive weight below preserves that ordering correctly (recent + high value -> higher score).
# recency_days is only used to restore a positive, human-readable value in the displayed table.

recency_days = inputs_pd['recency'].abs()

composite = round((recency_days * 3) + (inputs_pd['frequency'] * 2) + (inputs_pd['monetary_value'] * 1), 2)

inputs_pd['composite'] = composite
display(
  inputs_pd[['audienceid', 'recency', 'frequency', 'monetary_value', 'composite', 'cluster']]
    .assign(recency=recency_days)  # restore positive recency for readability
  )

In [0]:
# Absolute RFM score: unlike 'composite' above, these thresholds are fixed business rules rather than
# quantile bins recomputed from the current population on every run -- they're fixed ONCE (calibrated
# below from this dataset's own quintiles) so a customer's score stays stable across time/re-runs even
# if the overall customer distribution shifts -- only their own R/F/M values matter going forward.

# recency (days since last purchase, uninverted via recency_days from the cell above): fewer days = better
# thresholds recalibrated from this dataset's quintile edges (see "Diagnose Absolute Score Threshold
# Coverage" above -- the original generic 30/90/180/365-day cutoffs
# assumed a much faster purchase cadence than this ticket-sales business actually has, which pushed
# 60% of customers into the single lowest score; these quintile-based cutoffs spread scores evenly
recency_score = pd.cut(
  recency_days,
  bins=[-1, 484, 612, 900, 1210, np.inf],
  labels=[5, 4, 3, 2, 1]
  ).astype(int)

# frequency (unique purchase dates, already capped at 5 upstream): higher = better, used directly as the 1-5 score
frequency_score = inputs_pd['frequency'].clip(upper=5).astype(int)

# monetary value (avg spend per purchase, capped at value upstream): higher = better
monetary_score = pd.cut(
  inputs_pd['monetary_value'],
  bins=[-1, 160, 348, 660, 1460, np.inf],
  labels=[1, 2, 3, 4, 5]
  ).astype(int)


# combine into a single absolute RFM score (range 3-15); adjust weights here if one metric should matter more
inputs_pd['recency_score'] = recency_score
inputs_pd['frequency_score'] = frequency_score
inputs_pd['monetary_score'] = monetary_score
inputs_pd['rfm_score_absolute'] = recency_score + frequency_score + monetary_score

display(
  inputs_pd[['audienceid', 'recency', 'frequency', 'monetary_value',
             'recency_score', 'frequency_score', 'monetary_score', 'rfm_score_absolute', 'composite', 'cluster']]
    .assign(recency=recency_days)
  )

In [0]:
# sanity check: are the fixed thresholds above actually well-matched to this dataset's distribution,
# or are they generic assumptions? compare the population's actual quantiles against the fixed cutoffs.
print('recency_days quantiles (days since last purchase):')
print(recency_days.describe(percentiles=[.1, .25, .5, .75, .9]))

print('\nrecency_days quintile edges (for recalibrating the fixed bins below):')
_, quintile_edges = pd.qcut(recency_days, 5, retbins=True, duplicates='drop')
print([round(e) for e in quintile_edges])
print('\nrecency_score distribution (recalibrated fixed bins based on quantiles):')
print((inputs_pd['recency_score'].value_counts(normalize=True).sort_index() * 100).round(1).astype(str) + '%')

print('\n\nmonetary_value quantiles (avg spend per purchase, capped above):')
print(inputs_pd['monetary_value'].describe(percentiles=[.1, .25, .5, .75, .9]))
print('\nmonetary_value quintile edges (for recalibrating the fixed bins below):')
_, m_quintile_edges = pd.qcut(inputs_pd['monetary_value'], 5, retbins=True, duplicates='drop')
print([round(e, 2) for e in m_quintile_edges])
print('\nmonetary_score distribution (fixed bins:')
print((inputs_pd['monetary_score'].value_counts(normalize=True).sort_index() * 100).round(1).astype(str) + '%')

print('\n\nfrequency_score distribution (raw frequency capped at 5, used directly as score):')
print((inputs_pd['frequency_score'].value_counts(normalize=True).sort_index() * 100).round(1).astype(str) + '%')

In [0]:
f, axes = plt.subplots(nrows=1, ncols=4, squeeze=True, figsize=(42, 10))

axes[0].set_title('cluster')
sns.scatterplot(
  x='tsne_one',
  y='tsne_two',
  hue='cluster',
  palette=sns.color_palette('husl', inputs_pd[['cluster']].nunique()[0]),
  data=inputs_pd,
  alpha=0.4,
  ax = axes[0]
  )
axes[0].legend(loc='lower left', ncol=2, fancybox=True)

# chart the RFM scores
for i, metric in enumerate(['r_bin', 'f_bin', 'm_bin']):
  
  # unique values for this metric
  n = inputs_pd[['{0}'.format(metric)]].nunique()[0]
  
  # use metric name as chart title
  axes[i+1].set_title(metric)
  
  # define chart
  sns.scatterplot(
    x='tsne_one',
    y='tsne_two',
    hue='{0}'.format(metric),
    palette=sns.color_palette('coolwarm', n),
    data=inputs_pd,
    legend=False,
    alpha=0.4,
    ax = axes[i+1]
    )

The visualization of clusters relative to the RFM metrics helps us understand how clusters related to the metric values and the range of values associated with each cluster. We could perform more detailed analysis of the clusters to understand the distance between members and between the various clusters but a quick visual inspection is often sufficient for this kind of work.

In addition, we can extract the centroids of each cluster to more precisely understand how each relates to the RFM metrics. Please note that these centroids have exact positions that are captured in fractional values. But to help simplify the comparison of the clusters, we've rounded these up to the nearest integer value:

In [0]:
clusters = []

# for each cluster
for c in range(0, pipe[-1].n_clusters):
  # get integer values for metrics assocaited with each centroid
  centroids = np.abs(pipe[-1].cluster_centers_[c].round(0).astype('int')).tolist()
  # captuer cluster and centroid values
  clusters += [ [c] + centroids]

# convert details to dataframe
clusters_pd = pd.DataFrame(clusters, columns=['cluster','r_bin','m_bin'])

display(clusters_pd)

In [0]:
# descriptive labels based on centroid r_bin and m_bin values
# r_bin: 0 = lapsed, 4 = very recent | m_bin: 0 = low spend, 4 = high spend
# NOTE: cluster ids/centroids can shift between retrains (KMeans random init), so labels are
# derived dynamically from each cluster's rank *relative to the others* rather than hardcoded
# to a specific cluster id -- this keeps the mapping correct no matter how ids get reassigned.
r_median = clusters_pd['r_bin'].median()
m_median = clusters_pd['m_bin'].median()

def assign_label(row):
  is_recent = row['r_bin'] >= r_median
  is_high_value = row['m_bin'] >= m_median
  if is_recent and is_high_value:
    return 'Loyal High-Value'       # recent, high spenders — best active segment
  elif is_recent and not is_high_value:
    return 'Recent / Low-Value'     # fairly recent but lowest spenders — nurture to grow value
  elif not is_recent and is_high_value:
    return 'Lapsed High-Value'      # mostly lapsed but previously the biggest spenders — top win-back priority
  else:
    return 'Lost Customers'         # lapsed, low spenders — unlikely to return without intervention

clusters_pd['label'] = clusters_pd.apply(assign_label, axis=1)
display(clusters_pd[['cluster', 'r_bin', 'm_bin', 'label']])

In [0]:
# join descriptive labels back to customer-level data
customers_labeled = inputs_pd.merge(
  clusters_pd[['cluster', 'label']],
  on='cluster',
  how='left'
)

# display key columns, sorted by label for easy browsing
display(
  customers_labeled[['audienceid', 'recency', 'frequency', 'monetary_value', 'r_bin', 'f_bin', 'm_bin', 
  'recency_score', 'frequency_score', 'monetary_score', 'rfm_score_absolute', 'composite', 'cluster', 'label']]
  .assign(recency=customers_labeled['recency'].abs())  # restore positive recency for readability
  .sort_values('label')
  .reset_index(drop=True)
)

In [0]:
segment_counts = (
  customers_labeled
  .groupby(['cluster', 'label'])
  .agg(
    customers=('audienceid', 'count'),
    avg_recency=('recency', lambda x: x.abs().mean().round(0)),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary_value', 'mean')
  )
  .round({'avg_frequency': 2, 'avg_monetary': 2})
  .sort_values('cluster')
  .reset_index()
)
segment_counts['pct_of_total'] = (segment_counts['customers'] / segment_counts['customers'].sum() * 100).round(1).astype(str) + '%'

display(segment_counts[['cluster', 'label', 'customers', 'pct_of_total', 'avg_recency', 'avg_frequency', 'avg_monetary']])

In [0]:
# compare the distribution of both scores across the k=4 clusters -- shows how much spread
# (and outlier risk) exists within each segment, not just the segment's average
order = clusters_pd.sort_values('cluster')['label'].tolist()

f, axes = plt.subplots(nrows=1, ncols=2, squeeze=True, figsize=(20, 8))

sns.boxplot(x='label', y='composite', data=customers_labeled, order=order, palette='viridis', ax=axes[0])
axes[0].set_title('Composite RFM Score by Cluster')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)

sns.boxplot(x='label', y='rfm_score_absolute', data=customers_labeled, order=order, palette='viridis', ax=axes[1])
axes[1].set_title('Absolute RFM Score by Cluster')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()

In [0]:
# train a separate 6-cluster model purely for comparison against the production k=4 model
# (reuses the same recency/monetary quantile binning; does not replace the k=4 pipeline above)
model_k6 = KMeans(
  n_clusters=6,
  init='random',
  n_init=10000
  )

pipe_k6 = Pipeline(steps=[
  ('binnerize', col_trans),
  ('cluster', model_k6)
  ])

fitted_pipe_k6 = pipe_k6.fit(inputs_pd)
inputs_pd['cluster_k6'] = pipe_k6.predict(inputs_pd)

display(inputs_pd[['audienceid', 'r_bin', 'f_bin', 'm_bin', 'cluster', 'cluster_k6']])

In [0]:
clusters_k6 = []
for c in range(0, pipe_k6[-1].n_clusters):
  centroids = np.abs(pipe_k6[-1].cluster_centers_[c].round(0).astype('int')).tolist()
  clusters_k6 += [[c] + centroids]

clusters_k6_pd = pd.DataFrame(clusters_k6, columns=['cluster', 'r_bin', 'm_bin'])

# derive labels from *relative* rank so they're robust to cluster id reordering on retrain:
# recency split into 3 tiers (Lapsed / Mid-Recency / Recent), value split into 2 tiers
# (Low-Value / High-Value) -> 3 x 2 = 6 natural combinations for a 6-cluster model
clusters_k6_pd['recency_tier'] = pd.qcut(
  clusters_k6_pd['r_bin'].rank(method='first'), q=3, labels=['Lapsed', 'Mid-Recency', 'Recent']
  )
m_median_k6 = clusters_k6_pd['m_bin'].median()
clusters_k6_pd['value_tier'] = np.where(clusters_k6_pd['m_bin'] >= m_median_k6, 'High-Value', 'Low-Value')
clusters_k6_pd['label'] = clusters_k6_pd['recency_tier'].astype(str) + ' / ' + clusters_k6_pd['value_tier']

display(clusters_k6_pd[['cluster', 'r_bin', 'm_bin', 'label']])

In [0]:
# bring k=6 labels onto the customer level, then compare against the existing k=4 labels
customers_k6 = inputs_pd.merge(
  clusters_k6_pd[['cluster', 'label']].rename(columns={'cluster': 'cluster_k6', 'label': 'label_k6'}),
  on='cluster_k6', how='left'
  )

comparison = customers_labeled[['audienceid', 'label']].rename(columns={'label': 'label_k4'}).merge(
  customers_k6[['audienceid', 'label_k6']], on='audienceid', how='left'
  )

print('How each k=4 segment splits across the k=6 segments:')
crosstab = pd.crosstab(comparison['label_k4'], comparison['label_k6'])
display(crosstab)

# k=6 segment summary, for side-by-side comparison with the k=4 segment_counts table above
k6_segment_counts = (
  customers_k6
    .groupby(['cluster_k6', 'label_k6'])
    .agg(
      customers=('audienceid', 'count'),
      avg_recency=('recency', lambda x: x.abs().mean().round(0)),
      avg_frequency=('frequency', 'mean'),
      avg_monetary=('monetary_value', 'mean')
      )
    .round({'avg_frequency': 2, 'avg_monetary': 2})
    .reset_index()
    .sort_values('cluster_k6')
  )
k6_segment_counts['pct_of_total'] = (k6_segment_counts['customers'] / k6_segment_counts['customers'].sum() * 100).round(1).astype(str) + '%'

print('\nk=6 segment summary:')
display(k6_segment_counts[['cluster_k6', 'label_k6', 'customers', 'pct_of_total', 'avg_recency', 'avg_frequency', 'avg_monetary']])

In [0]:
# extend cell 45's customer-level output ('Customer Cluster Assignments') to also include
# the k=6 cluster assignment and label, for direct side-by-side comparison per customer
customers_labeled_k6 = (
  inputs_pd
    .merge(clusters_pd[['cluster', 'label']], on='cluster', how='left')
    .merge(
      clusters_k6_pd[['cluster', 'label']].rename(columns={'cluster': 'cluster_k6', 'label': 'label_k6'}),
      on='cluster_k6', how='left'
      )
  )

display(
  customers_labeled_k6[[
    'audienceid','recency', 'frequency', 'monetary_value',
    'r_bin', 'f_bin', 'm_bin', 'cluster', 'label', 'cluster_k6', 'label_k6'
  ]]
  .assign(recency=customers_labeled_k6['recency'].abs())  # restore positive recency for readability
  .sort_values('label')
  .reset_index(drop=True)
  )

In [0]:
# same comparison as "Compare RFM Scores Across Clusters" above, but for the k=6 comparison model
order_k6 = clusters_k6_pd.sort_values('cluster')['label'].tolist()

f, axes = plt.subplots(nrows=1, ncols=2, squeeze=True, figsize=(20, 8))

sns.boxplot(x='label_k6', y='composite', data=customers_labeled_k6, order=order_k6, palette='viridis', ax=axes[0])
axes[0].set_title('Composite RFM Score by Cluster (k=6)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)

sns.boxplot(x='label_k6', y='rfm_score_absolute', data=customers_labeled_k6, order=order_k6, palette='viridis', ax=axes[1])
axes[1].set_title('Absolute RFM Score by Cluster (k=6)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()

##Step 6: Persist Cluster Assignments

The previous steps in this notebook are intended to demonstrate how an RFM segmentation might be performed, but how might we operationalize the model for on-going work?  Every time we retrain our model, the centroids attached to a cluster id will vary.  If we wish to re-score customers periodically but keep cluster centroids the same between those runs, we need to persist and re-use our model.  This is made easy using the MLFlow model registry: 

In [0]:
# Unity Catalog requires a fully qualified 3-level name: catalog.schema.model_name
model_name = 'hbse.default.rfm_segmentation'

In [0]:
# to ensure this notebook runs in jobs
username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
_ = mlflow.set_experiment('/Users/{}/{}'.format(username, model_name))

In [0]:
with mlflow.start_run(run_name='deployment ready'):

  # unity catalog requires a signature (and input example) when registering a model
  sample_input = inputs_pd[['recency', 'frequency', 'monetary_value']].head(5)
  sample_output = fitted_pipe.predict(sample_input)
  signature = mlflow.models.infer_signature(sample_input, sample_output)

  mlflow.sklearn.log_model(
    fitted_pipe,
    'model',
    signature=signature,
    input_example=sample_input,
    registered_model_name=model_name
    )

We can then elevate our model to production status to indicate it is ready for use in an on-going ETL pipeline:

In [0]:
# connect to mlflow
client = mlflow.tracking.MlflowClient()

# identify latest model version in registry
latest_model_info = client.search_model_versions(f"name='{model_name}'")[0]
model_version = latest_model_info.version

# assign 'production' alias to the latest version
client.set_registered_model_alias(
  name=model_name,
  alias='production',
  version=model_version
  )

With our model persisted and elevated to production status, applying it to data is relatively easy:

In [0]:
# superseded by score_customers() in the next cell, which wraps this exact "load model directly +
# invert recency + predict" workaround into a reusable, general-purpose function. Kept here as a
# minimal one-off example against this notebook's already-computed rfm_metrics_cleansed.
loaded_model = mlflow.sklearn.load_model(f'models:/{model_name}@production')

scored_pd = rfm_metrics_cleansed.toPandas()
scored_pd['cluster'] = loaded_model.predict(
  scored_pd[['recency', 'frequency', 'monetary_value']].assign(recency=lambda d: d['recency'] * -1)
  )

display(spark.createDataFrame(scored_pd))

In [0]:
def score_customers(transactions_df, model_name='hbse.default.rfm_segmentation', model_alias='production', dynamic_recency_bins=False):
  """
  General-purpose scoring function: takes a raw transactions DataFrame with the same schema as
  hbse.default.njd_rfm_dataset (audienceid, emailaddress, saledate, total_sales, total_seats) and
  returns a pandas DataFrame with RFM metrics, composite score, absolute score, cluster
  assignment, and a descriptive cluster label for each customer -- reproducing Steps 1-2
  (metrics) and the composite/absolute score formulas, then calling the persisted production
  model directly for both the cluster id and its centroids (used to derive the label). This
  keeps the function fully self-contained: it never depends on this notebook's in-memory
  clusters_pd/pipe, so it never requires retraining anything to run.

  dynamic_recency_bins (default True): recompute the recency_score quintile cutoffs from THIS
  call's own data each time, instead of using fixed day cutoffs. This adapts automatically to a
  new dataset's purchase cadence (e.g. a faster- or slower-buying customer base), but it also
  means recency_score/rfm_score_absolute become relative to whatever batch is passed in -- the
  same customer could score differently across two separate calls, similar to 'composite'. Pass
  dynamic_recency_bins=False to use fixed, stable cutoffs instead (comparable across calls/time),
  matching the fixed-threshold design of "Calculate Absolute RFM Score" above. Only reliable with
  a reasonably large batch (a few hundred+ rows) -- quintiles from a handful of customers are
  not meaningful.
  """
  # Steps 1-2: consolidate transactions on date, then compute recency/frequency/monetary_value
  txns = (
    transactions_df
      .withColumn('total_revenue', fn.col('total_sales') * fn.col('total_seats'))
      .filter('total_revenue > 0.0')
      .withColumn('saledate', fn.to_date('saledate'))
      .groupBy('audienceid', 'saledate')
        .agg(fn.sum('total_revenue').alias('total_revenue'))
    )

  last_date = txns.groupBy().agg(fn.max('saledate').alias('lastdate'))

  rfm = (
    txns
      .crossJoin(last_date)
      .groupBy('audienceid')
        .agg(
          fn.min(fn.datediff('lastdate', 'saledate')).alias('recency'),
          fn.countDistinct('saledate').alias('frequency'),
          fn.round(fn.avg('total_revenue'), 2).alias('monetary_value')
          )
    )

  # cap outliers the same way as "Cap the Frequency & Monetary Value Metrics"
  rfm_capped = (
    rfm
      .withColumn('frequency', fn.expr("case when frequency > 5 then 5 else frequency end"))
      .withColumn('monetary_value', fn.expr("case when monetary_value > 20000 then 20000 else monetary_value end"))
    )

  scored_pd = rfm_capped.toPandas()

  # composite score: same fixed weights as "Calculate Composite RFM Score" (recency is raw/positive
  # here, so a negative weight reproduces that cell's inverted-recency * positive-weight formula)
  scored_pd['composite'] = round(
    (scored_pd['recency'] * 3) + (scored_pd['frequency'] * 2) + (scored_pd['monetary_value'] * 1), 2
    )

  # absolute score -- recency: either recalibrated from this batch's own quantiles (default), or the
  # same fixed thresholds as "Calculate Absolute RFM Score (Fixed Thresholds)" if dynamic_recency_bins=False
  if dynamic_recency_bins:
    _, recency_edges = pd.qcut(scored_pd['recency'], 5, retbins=True, duplicates='drop')
    recency_edges[0], recency_edges[-1] = -1, np.inf
    recency_labels = list(range(len(recency_edges) - 1, 0, -1))  # fewer days = higher score
    recency_score = pd.cut(scored_pd['recency'], bins=recency_edges, labels=recency_labels).astype(int)
  else:
    #recency_score = pd.cut(scored_pd['recency'], bins=[-1, 236, 338, 625, 950, np.inf], labels=[5, 4, 3, 2, 1]).astype(int)
    recency_score = pd.cut(scored_pd['recency'], bins=[-1, 484, 612, 900, 1210, np.inf], labels=[5, 4, 3, 2, 1]).astype(int)
  frequency_score = scored_pd['frequency'].clip(upper=5).astype(int)
  monetary_score = pd.cut(scored_pd['monetary_value'], bins=[-1, 160, 348, 660, 1460, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
  #monetary_score = pd.cut(scored_pd['monetary_value'], bins=[-1, 165, 379, 810, 3300, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
  scored_pd['recency_score'] = recency_score
  scored_pd['frequency_score'] = frequency_score
  scored_pd['monetary_score'] = monetary_score
  scored_pd['rfm_score_absolute'] = recency_score + frequency_score + monetary_score

  # cluster assignment via the persisted production model (invert recency to match training convention)
  loaded_model = mlflow.sklearn.load_model(f'models:/{model_name}@{model_alias}')
  scoring_input = scored_pd[['recency', 'frequency', 'monetary_value']].copy()
  scoring_input['recency'] = scoring_input['recency'] * -1
  scored_pd['cluster'] = loaded_model.predict(scoring_input)

  # derive descriptive labels directly from the LOADED model's own centroids (same logic as
  # "Assign Descriptive Labels to Clusters" above) -- this is what lets scoring skip retraining:
  # the label lookup is built from the persisted model, not from this notebook's clusters_pd/pipe
  kmeans_step = loaded_model[-1]
  centers = np.abs(kmeans_step.cluster_centers_.round(0).astype(int))
  clusters_lookup = pd.DataFrame(centers, columns=['r_bin', 'm_bin'])
  clusters_lookup['cluster'] = clusters_lookup.index

  r_median = clusters_lookup['r_bin'].median()
  m_median = clusters_lookup['m_bin'].median()

  def assign_label(row):
    is_recent = row['r_bin'] >= r_median
    is_high_value = row['m_bin'] >= m_median
    if is_recent and is_high_value:
      return 'Loyal High-Value'
    elif is_recent and not is_high_value:
      return 'Recent / Low-Value'
    elif not is_recent and is_high_value:
      return 'Lapsed High-Value'
    else:
      return 'Lost Customers'

  clusters_lookup['label'] = clusters_lookup.apply(assign_label, axis=1)
  scored_pd = scored_pd.merge(clusters_lookup[['cluster', 'label']], on='cluster', how='left')

  # z-scores for score columns (how far each customer deviates from the batch mean, in std units)
  for col in ['recency_score', 'frequency_score', 'monetary_score', 'rfm_score_absolute']:
    scored_pd[f'{col}_zscore'] = ((scored_pd[col] - scored_pd[col].mean()) / scored_pd[col].std()).round(4)

  return scored_pd


# Replace table here**: apply the function to the same source table used throughout this notebook
new_scores_pd = score_customers(spark.table("hbse.default.njd_rfm_dataset"))
display(new_scores_pd)

In [0]:
loyal_high_value_pd = new_scores_pd[new_scores_pd['label'] == 'Loyal High-Value'].copy()

# only pull audienceid (join key) + currentstm/priorstm/priorsgb from dim_aggregatefields
agg_fields_pd = (
  spark.table('kagr_njd.stage.dim_aggregatefields')
    .select('audienceid', 'currentstm', 'priorstm', 'priorsgb')
    .toPandas()
  )


sent_pd = (
  spark.sql("""
    SELECT
      g.audienceid,
      COUNT(*) AS total_sent
    	FROM kagr_njd.stage.sfmcsendjobs a
	  JOIN kagr_njd.stage.sfmcsent b on a.sendid = b.sendid
    JOIN kagr_njd.stage.rawaudience f ON b.subscriberkey = f.sourceaccountid
    JOIN kagr_njd.stage.audiencemapping g ON f.rawaudienceid = g.rawaudienceid
    WHERE a.subject NOT LIKE '%Test%' AND a.subject NOT LIKE '%test%'and a.emailname LIKE '%Devils%'
    GROUP BY g.audienceid
""")
  .toPandas()
)

# pull total_clicks and count of distinct email addresses per audienceid
# via sfmcclicks -> rawaudience -> audiencemapping (one row per audienceid, no fan-out)
clicks_pd = (
  spark.sql("""
    SELECT
      g.audienceid,
      COUNT(*) AS total_clicks,
      COUNT(DISTINCT c.emailaddress) AS total_emailaddresses
    FROM kagr_njd.stage.sfmcclicks c
    JOIN kagr_njd.stage.rawaudience f ON c.subscriberkey = f.sourceaccountid
    JOIN kagr_njd.stage.audiencemapping g ON f.rawaudienceid = g.rawaudienceid
    GROUP BY g.audienceid
    """)
    .toPandas()
  )

sales_funnel = (
  spark.sql("""
    select audienceid, sales_funnel from (
      select distinct audienceid, upperfunnel, midfunnel, lowerfunnel, 
        CASE 
          WHEN (upperfunnel IN ('1') AND lowerfunnel IS NULL AND midfunnel IS NULL) then 'Upper'
          WHEN (midfunnel IN ('1') AND lowerfunnel IS NULL) then 'Mid'
          WHEN (lowerfunnel IN ('1')) then 'Lower'
          ELSE 'Null'
          END AS Sales_Funnel
      from KAGR_HBSE.NJD_FLYWHEEL_DATASETS.FLY_MAT_AUDIENCE
    )
    WHERE Sales_Funnel NOT IN ('Null')
    """)
  .toPandas()
)

# align audienceid dtype on all sides before merging (avoids decimal/pandas dtype mismatches)
agg_fields_pd['audienceid'] = agg_fields_pd['audienceid'].astype(str)
sent_pd['audienceid'] = sent_pd['audienceid'].astype(str)
clicks_pd['audienceid'] = clicks_pd['audienceid'].astype(str)
sales_funnel['audienceid'] = sales_funnel['audienceid'].astype(str)
loyal_high_value_pd['audienceid'] = loyal_high_value_pd['audienceid'].astype(str)
new_scores_pd['audienceid'] = new_scores_pd['audienceid'].astype(str)

loyal_high_value_with_agg = (
  loyal_high_value_pd
    .merge(agg_fields_pd, on='audienceid', how='left')
    .merge(clicks_pd, on='audienceid', how='left')
  )

display(loyal_high_value_with_agg)

full_fan_with_agg = (
  new_scores_pd
    .merge(agg_fields_pd, on='audienceid', how='left')
    .merge(clicks_pd, on='audienceid', how='left')
    .merge(sent_pd, on='audienceid', how='left')
    .merge(sales_funnel, on='audienceid', how='left')
  )
full_fan_with_agg['sales_funnel'] = full_fan_with_agg['sales_funnel'].fillna('Null')
full_fan_with_agg = full_fan_with_agg.fillna(0)

full_fan_with_agg['click_rate'] = (full_fan_with_agg['total_clicks'] / full_fan_with_agg['total_sent']).replace([float('inf')], 0).fillna(0).round(4)

display(full_fan_with_agg)
  

In [0]:
# filter the score_customers() output (new_scores_pd, from the demo call in the cell above) down to
# Loyal High-Value customers, then pull in the ledgername(s) tied to each audienceid from
# primary_rev_consolidated (cell 6's output). A customer can have multiple ledgername values across
# their transaction history, so this can produce more than one row per audienceid.
loyal_high_value_pd = new_scores_pd[new_scores_pd['label'] == 'Loyal High-Value'].copy()

# distinct audienceid -> ledgername mapping
audience_ledger_pd = (
  primary_rev_consolidated
    .select('audienceid', 'ledgername')
    .distinct()
    .toPandas()
  )

# align audienceid dtype on both sides before merging (avoids decimal/pandas dtype mismatches)
audience_ledger_pd['audienceid'] = audience_ledger_pd['audienceid'].astype(str)
loyal_high_value_pd['audienceid'] = loyal_high_value_pd['audienceid'].astype(str)

loyal_high_value_with_ledger = loyal_high_value_pd.merge(
  audience_ledger_pd,
  on='audienceid',
  how='left'
  )

display(loyal_high_value_with_ledger)

In [0]:
# alternative to score_customers(spark.table(...)): run a query directly and skip persisting a
# table entirely. Edit the SQL below (source tables, date filter, WHERE clauses, etc.) to point
# at whatever data you want -- it just needs to end up with columns:
# audienceid, emailaddress, saledate, total_seats, total_sales
# (this template reuses the same joins as the NJD_RFM_dataset query, without the CREATE TABLE wrapper)

query = """
WITH base_data AS (
  SELECT
    a.*, b.*
  FROM
    (
    SELECT
        g.audienceid,
        a.sendid,
        a.subject,
        lower(a.emailname || 'Devils') AS emailname,
        a.emailname AS origemail,
        b.emailaddress,
        b.subscriberkey,
        to_date(b.eventdate) as senddate,
        senttime
    FROM
        kagr_njd.stage.sfmcsendjobs a
        JOIN kagr_njd.stage.sfmcsent b ON a.sendid = b.sendid
        JOIN kagr_njd.stage.rawaudience f ON b.subscriberkey = f.sourceaccountid
        JOIN kagr_njd.stage.audiencemapping g USING (rawaudienceid)
    WHERE a.subject NOT LIKE '%Test%' AND a.subject NOT LIKE '%test%' AND a.emailname LIKE '%Devils%'
    ) a
  JOIN
    (
    SELECT
     audienceid as audienceid_key, to_date(c.saledate) as saledate,
      c.ledgername, c.tenure, count(c.seatnumber) as total_seats, sum(c.purchaseprice) as primary_cost, sum(c.resaleatp) as secondary_cost, 
      (COALESCE(c.purchaseprice, 0) + COALESCE(c.resaleatp, 0)) as total_cost
  from
      kagr_hbse.stage.vw_njd_seatmapgamesummary_enhanced c
      join kagr_njd.stage.audiencemapping d on c.rawaudienceid = d.rawaudienceid
  where
      compname = 'Not Comp'
      and ledgercode not in ('BKS', 'FSR', 'HSR')
  group by audienceid, ledgername, tenure, saledate, purchaseprice, resaleatp) b
  ON a.audienceid = b.audienceid_key AND a.senddate = b.saledate
  INNER JOIN
    (
    SELECT audienceid, eversgb
      FROM kagr_njd.stage.dim_aggregatefields
      WHERE eversgb = 1
    ) c
  ON a.audienceid = c.audienceid
  WHERE
    saledate >= '2024-07-01 00:00:00.000'
)
SELECT
  audienceid, emailaddress, saledate,
  SUM(total_seats) AS total_seats,
  CAST(SUM(total_cost) AS DOUBLE) AS total_sales
FROM base_data
GROUP BY audienceid, emailaddress, saledate
"""

new_transactions_df = spark.sql(query)

new_scores_from_query_pd = score_customers(new_transactions_df)
display(new_scores_from_query_pd)

In [0]:
# show the recency bin edges actually applied to new_scores_from_query_pd (the cell above) -- derived
# from the returned data itself (min/max recency per recency_score), so it reflects exactly what was
# used, whether that came from quantiles (dynamic_recency_bins=True) or fixed day cutoffs (False).
# keep this flag in sync with whatever was passed to score_customers(...) in the cell above.
dynamic_recency_bins = True

if dynamic_recency_bins:
  recency_bins_used_query = (
    new_scores_from_query_pd
      .groupby('recency_score')['recency']
      .agg(min_recency='min', max_recency='max', customers='count')
      .sort_index(ascending=False)
    )
  print('Recency bin edges applied above (dynamic_recency_bins=True, recalibrated from this batch):')
  display(recency_bins_used_query)
else:
  print('dynamic_recency_bins=False -- fixed cutoffs were used instead. See cell 52 for values.')

Of course, to make use of these clusters, we'll want access to descriptive information about what each cluster represents.  Typically, the marketing team will assign friendly labels to each cluster that explain what they represent in easy to understand terms.  For our purposes, we'll just persist the centroid information extracted in the last step.  These data could then be joined with the output of the previous cell to provide friendly labels for each cluster assignment:

In [0]:
_ = (
  spark
    .createDataFrame(clusters_pd)
    .withColumn( # more typically, a friendly name would be assigned by marketing
      'label', 
      fn.expr("concat('Cluster ', cluster, ': r=', r_bin, ', m=', m_bin )")
      )
    .write
      .format('delta')
      .mode('overwrite')
      .option('overwriteSchema','true')
      .saveAsTable('rfm_clusters')
  )

display(spark.table('rfm_clusters'))